# E8: замороженный контекст B5 — Cityscapes, Colab

**Никакого обучения B0/B5.** 500 Cityscapes val: Frankfurt 267 для банка kNN, Lindau/Münster 233 для оценки. ACDC пока не используется.

1. Открой этот файл через https://colab.research.google.com → **Файл → Загрузить блокнот**.
2. **Среда выполнения → Сменить среду выполнения → T4 GPU** (или другой доступный GPU).
3. Сохрани копию ноутбука на Drive. Запускай ячейки сверху вниз.
4. В своей [папке данных](https://drive.google.com/drive/folders/1O8_9tp_lj4ZHjKisqcUszGtdvFvXQp74?usp=sharing) найди `val.zip` и `gtFine_trainvaltest.zip`. Если папка чужая, добавь ярлык в «Мой диск». В ячейке 2 укажи пути к двум архивам. `test.zip` не нужен.

Ноутбук содержит весь код; клонировать GitHub и входить в него не нужно. Источник: присланные пользователем common/E8, минимальная подготовка entropy+kNN. Банк kNN имеет собственный фиксированный seed и сохраняется; это **новый запуск**, не точное воспроизведение исторических чисел E3/E5.

Результаты, логи, опоры kNN и резервные копии кэша сохраняются в отдельную папку на Drive. Распаковка изображений и GPU-вычисления происходят на локальном диске Colab. Исходные архивы не меняются.


In [ ]:
# 1. Окружение. PyTorch/CUDA не переустанавливаем.
import os, sys, subprocess
os.environ['USE_TF'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.46.3', 'cityscapesscripts==2.2.4', 'scikit-learn', 'tqdm'], check=True)
import torch
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения → Сменить среду выполнения'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Укажи папку с архивами. Обычно нужно поменять ТОЛЬКО эту строку.
from pathlib import Path
ARCHIVE_DIR = Path('/content/drive/MyDrive/Cityscapes')  # <-- твоя папка или ярлык
IMAGE_ZIP = ARCHIVE_DIR / 'val.zip'
GT_ZIP = ARCHIVE_DIR / 'gtFine_trainvaltest.zip'

# Если архив называется иначе, поменяй IMAGE_ZIP/GT_ZIP выше.
for p in [IMAGE_ZIP, GT_ZIP]:
    assert p.is_file(), f'Не найден {p}. Найди архив в панели Files → drive → MyDrive и скопируй путь.'

RUN_NAME = 'e8_cityscapes_v1'  # для нового независимого запуска используй другое имя
SAVE = Path('/content/drive/MyDrive') / RUN_NAME
WORK = Path('/content') / RUN_NAME
EXP = WORK / 'experiments'
DATA = WORK / 'data'
CACHE = WORK / 'cache'
RESULTS = SAVE / 'results'
for p in [SAVE, EXP, DATA, CACHE, RESULTS]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(E8_DATA=str(DATA), E8_CACHE=str(CACHE), E8_RESULTS=str(RESULTS), E8_BACKUP=str(SAVE/'backups'))
print('Результаты:', SAVE)


## Распаковка только validation
Поддерживаются `val/<город>/...` и `leftImg8bit/val/<город>/...` в архиве изображений. Из разметки берём только `gtFine/val/*/*_labelIds.png`. Полный train/test не распаковывается.

In [ ]:
# 3. Извлечение и проверка 500 пар.
import zipfile, shutil
from collections import Counter

def unpack_cityscapes(archive, image):
    count = 0
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            parts = Path(member.filename).parts
            suffix = '_leftImg8bit.png' if image else '_gtFine_labelIds.png'
            if member.is_dir() or not member.filename.endswith(suffix) or 'val' not in parts:
                continue
            k = parts.index('val')
            tail = parts[k:]
            if len(tail) != 3 or '..' in tail: continue
            branch = 'leftImg8bit' if image else 'gtFine'
            out = DATA/'cityscapes'/branch/Path(*tail)
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != member.file_size:
                tmp = out.with_suffix(out.suffix+'.part')
                with z.open(member) as src, tmp.open('wb') as dst: shutil.copyfileobj(src,dst)
                tmp.replace(out)
            count += 1
    return count

print('Изображений:', unpack_cityscapes(IMAGE_ZIP, True))
print('Масок:', unpack_cityscapes(GT_ZIP, False))
images = sorted((DATA/'cityscapes/leftImg8bit/val').glob('*/*_leftImg8bit.png'))
counts = Counter(p.parent.name for p in images)
assert dict(counts) == {'frankfurt':267, 'lindau':59, 'munster':174}, counts
for p in images:
    gt = DATA/'cityscapes/gtFine/val'/p.parent.name/p.name.replace('_leftImg8bit.png','_gtFine_labelIds.png')
    assert gt.is_file(), f'Отсутствует маска {gt}'
print('500 пар OK:', dict(counts))


In [ ]:
# 4. Создание автономных скриптов (никакие файлы репозитория не меняются).
import json, hashlib
SOURCES = {'common.py': 'import os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom transformers import SegformerForSemanticSegmentation\nfrom cityscapesscripts.helpers.labels import labels\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ["E8_DATA"])\nCACHE = Path(os.environ["E8_CACHE"])\nRESULTS = Path(os.environ["E8_RESULTS"])\n\nSIZE = (512, 1024)\nNUM_CLASSES = 19\nIGNORE = 255\n\nMODELS = {\n    "b0": "nvidia/segformer-b0-finetuned-cityscapes-512-1024",\n    "b5": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",\n}\n\n# frankfurt is used to fit scores/routers, the rest is test only\nDOMAINS = ["cs_fit", "cs_test", "fog", "night", "rain", "snow"]\n\n_LUT = np.full(256, IGNORE, np.uint8)\nfor l in labels:\n    if 0 <= l.id < 256 and l.trainId not in (-1, 255):\n        _LUT[l.id] = l.trainId\n\n\ndef list_domain(domain):\n    """Returns sorted (image, gt) path pairs for a domain."""\n    if domain.startswith("cs_"):\n        cities = ["frankfurt"] if domain == "cs_fit" else ["lindau", "munster"]\n        imgs = [p for c in cities for p in (DATA / "cityscapes/leftImg8bit/val" / c).glob("*.png")]\n        gt = lambda p: Path(str(p).replace("leftImg8bit", "gtFine", 1)\n                            .replace("_leftImg8bit.png", "_gtFine_labelIds.png"))\n    else:\n        imgs = list((DATA / "acdc/rgb_anon" / domain / "val").rglob("*_rgb_anon.png"))\n        gt = lambda p: Path(str(p).replace("rgb_anon", "gt", 1)\n                            .replace("_rgb_anon.png", "_gt_labelTrainIds.png"))\n    return [(p, gt(p)) for p in sorted(imgs)]\n\n\ndef image_tensor(image):\n    # Tensor bilinear antialiased resize, matching the original torchvision v2 intent.\n    import torch.nn.functional as F\n    a = np.array(image, dtype=np.uint8, copy=True)\n    x = torch.from_numpy(a).permute(2,0,1).float().div(255)\n    x = F.interpolate(x[None], size=SIZE, mode="bilinear", align_corners=False, antialias=True)[0]\n    mean = torch.tensor([.485,.456,.406])[:,None,None]\n    std = torch.tensor([.229,.224,.225])[:,None,None]\n    return (x-mean)/std\n\n\nclass SegSet(Dataset):\n    def __init__(self, domain):\n        self.domain = domain\n        self.items = list_domain(domain)\n        self.img_tf = image_tensor\n\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        img_path, gt_path = self.items[i]\n        gt = np.array(Image.open(gt_path))\n        if self.domain.startswith("cs_"):\n            gt = _LUT[gt]\n        gt = Image.fromarray(gt).resize(SIZE[::-1], Image.NEAREST)\n        return self.img_tf(Image.open(img_path).convert("RGB")), torch.from_numpy(np.array(gt)), img_path.stem\n\n\ndef load_model(name, device="cuda"):\n    model = SegformerForSemanticSegmentation.from_pretrained(MODELS[name])\n    return model.half().to(device).eval()\n\n\ndef confusion(pred, gt):\n    m = gt != IGNORE\n    return np.bincount(NUM_CLASSES * gt[m].astype(np.int64) + pred[m],\n                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)\n\n\ndef miou(conf):\n    iou = np.diag(conf) / (conf.sum(0) + conf.sum(1) - np.diag(conf) + 1e-9)\n    return 100 * iou[conf.sum(1) > 0].mean()\n\n\nTILE = 64  # 512x1024 -> 8x16 tiles\n\n\ndef load(domain, *keys):\n    """Yields cached arrays of every image in a domain, in a fixed order."""\n    for p in sorted((CACHE / domain).glob("*.npz")):\n        with np.load(p) as z:\n            yield [z[k] for k in keys]\n\n\ndef tile_sum(x):\n    h, w = x.shape\n    return x.reshape(h // TILE, TILE, w // TILE, TILE).sum((1, 3))\n\n\ndef gain(gt, pred_b0, pred_b5):\n    """+1 where only B5 is right, -1 where only B0 is right, 0 elsewhere or on ignore."""\n    return ((pred_b5 == gt).astype(np.int8) - (pred_b0 == gt)) * (gt != IGNORE)\n\n\ndef top_tiles(scores, budget):\n    """Boolean mask of the top `budget` share of tiles by score, over the whole domain."""\n    order = np.argsort(-scores.ravel(), kind="stable")\n    m = np.zeros(scores.size, bool)\n    m[order[:round(budget * scores.size)]] = True\n    return m.reshape(scores.shape)\n\n\ndef route_confs(domain, scores, budgets):\n    """Per-image confusions (budgets x images x C x C) when top_tiles get B5."""\n    masks = [top_tiles(scores, b) for b in budgets]\n    confs = np.zeros((len(budgets), len(scores), NUM_CLASSES, NUM_CLASSES), np.int64)\n    for i, (gt, b0, b5) in enumerate(load(domain, "gt", "pred_b0", "pred_b5")):\n        for j, m in enumerate(masks):\n            px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n            confs[j, i] = confusion(np.where(px, b5, b0), gt)\n    return confs\n\n\ndef route_curve(domain, scores, budgets):\n    """mIoU per budget, see route_confs."""\n    return [miou(c.sum(0)) for c in route_confs(domain, scores, budgets)]\n', 'e2_confidence.py': '"""E2: standard confidence scores of B0 as routing signals.\n\nPixel AUROC (logit resolution, sampled pixels) for two targets:\n  err  - B0 is wrong\n  help - B0 is wrong and B5 is right\nTile routing: mean score per tile, % of the B5-B0 gap closed (as in E1).\n"""\nimport json\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_auc_score\n\nfrom common import DOMAINS, IGNORE, RESULTS, TILE, load, route_curve\n\nBUDGETS = [0.05, 0.1, 0.2, 0.3]\nPIXELS_PER_IMAGE = 2000\nSTRIDE = 4  # logits are at 1/4 resolution\n\n\ndef uncertainty(logits):\n    """Per-pixel scores, higher = less confident."""\n    p = logits.softmax(0)\n    top2 = p.topk(2, dim=0).values\n    return {\n        "entropy": -(p * p.clamp_min(1e-12).log()).sum(0),\n        "msp": 1 - top2[0],\n        "margin": top2[1] - top2[0],\n        "energy": -logits.logsumexp(0),\n    }\n\n\ndef main():\n    base = json.loads((RESULTS / "e0.json").read_text())\n    oracle = json.loads((RESULTS / "e1.json").read_text())\n    rng = np.random.default_rng(0)\n    auroc, closed = {}, {}\n\n    for domain in DOMAINS:\n        tiles, pix, err, helps = {}, {}, [], []\n        for gt, b0, b5, logits in load(domain, "gt", "pred_b0", "pred_b5", "logits_b0"):\n            scores = uncertainty(torch.from_numpy(logits).float())\n            for k, s in scores.items():\n                tiles.setdefault(k, []).append(F.avg_pool2d(s[None], TILE // STRIDE)[0].numpy())\n\n            gt, b0, b5 = (a[STRIDE // 2::STRIDE, STRIDE // 2::STRIDE] for a in (gt, b0, b5))\n            valid = np.flatnonzero(gt != IGNORE)\n            idx = rng.choice(valid, min(PIXELS_PER_IMAGE, valid.size), replace=False)\n            for k, s in scores.items():\n                pix.setdefault(k, []).append(s.numpy().ravel()[idx])\n            wrong = b0.ravel()[idx] != gt.ravel()[idx]\n            err.append(wrong)\n            helps.append(wrong & (b5.ravel()[idx] == gt.ravel()[idx]))\n\n        err, helps = np.concatenate(err), np.concatenate(helps)\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        auroc[domain], closed[domain] = {}, {}\n        for k in tiles:\n            s = np.concatenate(pix[k])\n            auroc[domain][k] = [round(roc_auc_score(err, s), 3), round(roc_auc_score(helps, s), 3)]\n            curve = route_curve(domain, np.stack(tiles[k]), BUDGETS)\n            closed[domain][k] = [round(100 * (m - b0_miou) / (b5_miou - b0_miou), 1) for m in curve]\n\n    print(f"{\'domain\':8} {\'method\':8} {\'AUC err\':>8} {\'AUC help\':>9}" + "".join(f"{b:>7.0%}" for b in BUDGETS))\n    for domain in DOMAINS:\n        for k, (a_err, a_help) in auroc[domain].items():\n            print(f"{domain:8} {k:8} {a_err:8.3f} {a_help:9.3f}" + "".join(f"{c:7.1f}" for c in closed[domain][k]))\n        print(f"{domain:8} {\'oracle\':8} {\'\':8} {\'\':9}" + "".join(f"{c:7.1f}" for c in oracle[domain]["oracle_tile"][:len(BUDGETS)]))\n\n    (RESULTS / "e2.json").write_text(json.dumps({"budgets": BUDGETS, "auroc": auroc, "closed": closed}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'e8_cityscapes.py': '"""E8: gate for the adapter idea - is a frozen stage-3 context good enough for B5?\n\nB5 sees the whole frame, but in the chosen stages only the tokens of routed tiles are updated\nby the transformer blocks. Every other token keeps its stage-entry value for all blocks, so it\nonly serves as keys/values ("frozen context"). Here the entry values come from the real B5\n(an upper bound for any adapter that maps B0 tokens into B5 token space).\n\nQuality only: the whole frame is still computed, only the write-back is masked. The saving is\ncounted analytically for now (blocks run on routed queries only; context costs K/V projections),\nreal FLOPs and latency are measured later.\n\nStages (0-based): 2 = stride 16, 40 blocks, most of B5\'s encoder compute; 3 = stride 32.\nVariants: "s3" freezes stage 2 only, "s3+s4" freezes stages 2 and 3. Stages 0-1 are run on the\nwhole frame in this experiment (known simplification, about a fifth of the encoder cost).\n\nTiles are selected as in E5 (domain-wide ranking at 10% and 20%): `rank` (entropy + knn) and the oracle.\nReported as % of the B5-B0 gap closed, next to:\n  full  - B5 on the full frame inside the routed tiles (the E5 "full" column, the target)\n  agree - pixel agreement with that full-frame B5 prediction inside the routed tiles\nGate: s3 should stay close to `full` on ACDC. For reference E5 ctx64 (cost x0.9) closed 42 / 17 / 35 / 60\n(fog / night / rain / snow) against 61 / 30 / 59 / 59 for full.\n"""\nimport json\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom common import (CACHE, DOMAINS, NUM_CLASSES, RESULTS, SIZE, TILE, SegSet, confusion, gain, load,\n                    load_model, miou, tile_sum, top_tiles)\ndef rank(scores):\n    return scores.ravel().argsort().argsort().reshape(scores.shape)\n\nDOMAINS = ["cs_fit", "cs_test"]\n\nBUDGETS = [0.1, 0.2]\nSTAGES = {"s3": [2], "s3+s4": [2, 3]}\n\n\nclass Frozen(nn.Module):\n    """Wraps a SegformerLayer: tokens outside state.mask are reset to the stage entry value."""\n\n    def __init__(self, layer, state, first):\n        super().__init__()\n        self.layer, self.state, self.first = layer, state, first\n\n    def forward(self, hidden_states, *args, **kwargs):\n        s = self.state\n        if self.first:\n            s.entry = hidden_states\n        out = self.layer(hidden_states, *args, **kwargs)\n        if s.mask is None:\n            return out\n        return (torch.where(s.mask, out[0], s.entry),) + tuple(out[1:])\n\n\ndef install(model, stages=(2, 3)):\n    """Wrap every block of the given stages, returns {stage: state}."""\n    state = {st: SimpleNamespace(mask=None, entry=None) for st in stages}\n    blocks = model.segformer.encoder.block\n    for st in stages:\n        for i, layer in enumerate(blocks[st]):\n            blocks[st][i] = Frozen(layer, state[st], i == 0)\n    return state\n\n\ndef token_mask(m, stage):\n    """(8, 16) bool tile mask -> (1, N, 1) bool mask over the tokens of a stage (stride 4 * 2**stage)."""\n    k = TILE // (4 * 2 ** stage)\n    t = torch.from_numpy(m).cuda().repeat_interleave(k, 0).repeat_interleave(k, 1)\n    return t.reshape(1, -1, 1)\n\n\n@torch.inference_mode()\ndef predict(model, state, img, masks):\n    """B5 prediction on the full frame where only `masks` {stage: token mask} tokens are updated."""\n    for st, s in state.items():\n        s.mask = masks.get(st)\n    try:\n        logits = model(pixel_values=img[None].cuda().half()).logits\n        return F.interpolate(logits.float(), SIZE, mode="bilinear", align_corners=False).argmax(1)[0].byte().cpu().numpy()\n    finally:\n        for s in state.values():\n            s.mask = None\n            s.entry = None\n\n\n\ndef sanity(model, state):\n    """An all-true mask must reproduce the cached full-frame B5 prediction."""\n    img = SegSet("cs_fit")[0][0]\n    (cached,) = next(load("cs_fit", "pred_b5"))\n    full = {st: token_mask(np.ones((SIZE[0] // TILE, SIZE[1] // TILE), bool), st) for st in state}\n    agree = (predict(model, state, img, full) == cached).mean()\n    assert agree > 0.999, f"all-true mask does not reproduce B5, agreement {agree:.4f}"\n\n\ndef main():\n    model = load_model("b5")\n    state = install(model)\n    sanity(model, state)\n\n    base = json.loads((RESULTS / "e0_cityscapes.json").read_text())\n    feats = np.load(CACHE / "e8_tiles_cityscapes.npz")\n    report_path = RESULTS / "e8_cityscapes.json"\n    report = json.loads(report_path.read_text()) if report_path.exists() else {}\n\n    print(f"{\'domain\':8} {\'method\':8} {\'budget\':>6} {\'full\':>7}"\n          + "".join(f"{n:>9}" for n in STAGES) + "".join(f"{f\'agree {n}\':>14}" for n in STAGES))\n    for domain in DOMAINS:\n        if domain in report:\n            print(domain, "уже завершён, пропускаем", flush=True)\n            continue\n        dataset = SegSet(domain)\n        assert [p.stem for p, _ in dataset.items] == [p.stem for p in sorted((CACHE / domain).glob("*.npz"))]\n\n        entropy, knn = feats[f"{domain}__entropy__mean"], feats[f"{domain}__knn__mean"]\n        gains = np.stack([tile_sum(gain(*a)) for a in load(domain, "gt", "pred_b0", "pred_b5")])\n        scores = {"rank": rank(entropy) + rank(knn), "oracle": gains}\n        masks = {(k, b): top_tiles(s, b) for k, s in scores.items() for b in BUDGETS}\n\n        zeros = lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)\n        full = {key: zeros() for key in masks}\n        conf = {(*key, n): zeros() for key in masks for n in STAGES}\n        agree = {(*key, n): np.zeros(2, np.int64) for key in masks for n in STAGES}\n\n        started = __import__("time").monotonic()\n        loader = DataLoader(dataset, batch_size=None, num_workers=2)\n        for i, ((img, _, _), (gt, b0, b5)) in enumerate(zip(loader, load(domain, "gt", "pred_b0", "pred_b5"))):\n            for key, m in masks.items():\n                px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n                full[key] += confusion(np.where(px, b5, b0), gt)\n                for n, stages in STAGES.items():\n                    pred = predict(model, state, img, {st: token_mask(m[i], st) for st in stages})\n                    conf[(*key, n)] += confusion(np.where(px, pred, b0), gt)\n                    agree[(*key, n)] += [(pred == b5)[px].sum(), px.sum()]\n\n            if (i+1) % 5 == 0 or i+1 == len(dataset):\n                elapsed = __import__("time").monotonic()-started\n                eta = elapsed/(i+1)*(len(dataset)-i-1)/60\n                print(f"E8 {domain}: {i+1}/{len(dataset)}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин", flush=True)\n\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        closed = lambda c: round(100 * (miou(c) - b0_miou) / (b5_miou - b0_miou), 1)\n        report[domain] = {}\n        for key in masks:\n            row = {"full": closed(full[key])}\n            row.update({n: closed(conf[(*key, n)]) for n in STAGES})\n            row.update({f"agree_{n}": round(100 * agree[(*key, n)][0] / agree[(*key, n)][1], 1) for n in STAGES})\n            report[domain][f"{key[0]}@{key[1]}"] = row\n            print(f"{domain:8} {key[0]:8} {key[1]:6.0%} {row[\'full\']:7.1f}"\n                  + "".join(f"{row[n]:9.1f}" for n in STAGES)\n                  + "".join(f"{row[f\'agree_{n}\']:14.1f}" for n in STAGES))\n\n        temp = report_path.with_suffix(".json.part")\n        temp.write_text(json.dumps(report, indent=2))\n        temp.replace(report_path)\n        print("Сохранён домен:", domain, flush=True)\n\n    print("Результат:", report_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'runner.py': '"""Cityscapes-only preparation and E8; frozen B0/B5. No ACDC tuning."""\nimport os\nos.environ[\'USE_TF\'] = \'0\'\nimport argparse, gc, hashlib, json, platform, tarfile, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load, load_model, confusion, miou\n\nDOMAINS = [\'cs_fit\', \'cs_test\']\nSIGNATURE = \'e8-cs-colab-v1\'\n\ndef write_json(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.part\')\n    tmp.write_text(json.dumps(data, indent=2))\n    tmp.replace(path)\n\ndef checkpoint(domain):\n    backup = os.environ.get(\'E8_BACKUP\')\n    if not backup:\n        return\n    dest = Path(backup); dest.mkdir(parents=True, exist_ok=True)\n    name = dest / f\'cache_{domain}.tar\'\n    tmp = name.with_suffix(\'.tar.part\')\n    with tarfile.open(tmp, \'w\') as tar:\n        tar.add(CACHE / domain, arcname=domain)\n    tmp.replace(name)\n    print(\'Кэш сохранён на Drive:\', name, flush=True)\n\ndef valid_cache(path):\n    if not path.exists():\n        return False\n    with np.load(path, allow_pickle=False) as z:\n        assert set([\'gt\', \'pred_b0\', \'pred_b5\', \'logits_b0\', \'feat_b0\']) <= set(z.files)\n        assert all(z[k].shape == SIZE for k in [\'gt\', \'pred_b0\', \'pred_b5\'])\n        assert z[\'logits_b0\'].shape == (19, 128, 256)\n        assert z[\'feat_b0\'].shape == (256, 64, 128)\n    return True\n\n@torch.inference_mode()\ndef cache():\n    models = [load_model(n) for n in [\'b0\', \'b5\']]\n    captured = {}\n    handle = models[0].decode_head.classifier.register_forward_pre_hook(lambda m, x: captured.update(f=x[0]))\n    report = {}\n    try:\n        for domain in DOMAINS:\n            data = SegSet(domain)\n            assert len(data) == (267 if domain == \'cs_fit\' else 233)\n            folder = CACHE / domain; folder.mkdir(parents=True, exist_ok=True)\n            confs = [np.zeros((19,19), np.int64) for _ in range(2)]\n            start = time.monotonic()\n            for i in range(len(data)):\n                img, gt, name = data[i]\n                path = folder / f\'{name}.npz\'\n                if not valid_cache(path):\n                    x = img[None].cuda().half()\n                    l0 = models[0](pixel_values=x).logits\n                    feat = F.avg_pool2d(captured[\'f\'], 2)[0].cpu().numpy()\n                    preds = []\n                    for logits in [l0, models[1](pixel_values=x).logits]:\n                        preds.append(F.interpolate(logits.float(), SIZE, mode=\'bilinear\', align_corners=False).argmax(1)[0].byte().cpu().numpy())\n                    tmp = path.with_suffix(\'.npz.part\')\n                    with tmp.open(\'wb\') as f:\n                        np.savez_compressed(f, gt=gt.numpy(), pred_b0=preds[0], pred_b5=preds[1], logits_b0=l0[0].cpu().numpy(), feat_b0=feat)\n                    tmp.replace(path)\n                with np.load(path) as z:\n                    for j, k in enumerate([\'pred_b0\', \'pred_b5\']):\n                        confs[j] += confusion(z[k], z[\'gt\'])\n                if (i+1) % 10 == 0 or i+1 == len(data):\n                    print(f\'E0 {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\', flush=True)\n            report[domain] = dict(zip([\'b0\',\'b5\'], [float(miou(c)) for c in confs]))\n            write_json(RESULTS / \'e0_cityscapes.json\', report)\n            checkpoint(domain)\n    finally:\n        handle.remove()\n    print(report, flush=True)\n\n@torch.inference_mode()\ndef scores():\n    """Only entropy + kNN required by E8. Bank fitted solely on cs_fit."""\n    from e2_confidence import uncertainty\n    from common import IGNORE\n    reference = CACHE / \'e8_knn_bank.npy\'\n    if not reference.exists():\n        rng = np.random.default_rng(0)\n        samples = []\n        for gt, feat in load(\'cs_fit\', \'gt\', \'feat_b0\'):\n            y = gt[4::8,4::8].ravel()\n            valid = np.flatnonzero(y != IGNORE)\n            idx = rng.choice(valid, min(500, len(valid)), replace=False)\n            samples.append(feat.reshape(256,-1).T[idx].astype(np.float32))\n        x = np.concatenate(samples)\n        # Separate seeded RNG: exact bank saved, not claimed identical to old E3.\n        idx = np.random.default_rng(0).permutation(len(x))[:20000]\n        bank = F.normalize(torch.from_numpy(x[idx]), dim=1).numpy()\n        np.save(reference, bank)\n    bank = torch.from_numpy(np.load(reference)).cuda().float()\n    assert bank.shape == (20000,256)\n    output = {}\n    for domain in DOMAINS:\n        arrays = {\'entropy\':[], \'knn\':[]}\n        count = len(SegSet(domain))\n        for i, (logits, feat) in enumerate(load(domain, \'logits_b0\',\'feat_b0\')):\n            entropy = uncertainty(torch.from_numpy(logits).float())[\'entropy\']\n            entropy = F.avg_pool2d(entropy[None],2)[0].numpy()\n            f = F.normalize(torch.from_numpy(feat.astype(np.float32)).cuda().flatten(1).T, dim=1)\n            distances = []\n            for chunk in f.split(512):\n                distances.append((1-(chunk @ bank.T).topk(10,dim=1).values.mean(1)).cpu())\n            knn = torch.cat(distances).numpy().reshape(64,128)\n            for name, m in [(\'entropy\',entropy),(\'knn\',knn)]:\n                # 8x8 feature cells per 64x64 tile.\n                arrays[name].append(m.reshape(8,8,16,8).mean((1,3)))\n            if (i+1)%10==0 or i+1==count:\n                print(f\'kNN {domain}: {i+1}/{count}\', flush=True)\n        for k,v in arrays.items():\n            output[f\'{domain}__{k}__mean\'] = np.stack(v)\n    path = CACHE / \'e8_tiles_cityscapes.npz\'\n    np.savez(path, **output)\n    write_json(RESULTS/\'knn_config.json\', {\'fit\':\'cs_fit only\', \'bank\':20000, \'k\':10, \'sampling_seed\':0,\n        \'bank_sha256\':hashlib.sha256(reference.read_bytes()).hexdigest(),\n        \'note\':\'Minimal E3 alternative; saved bank differs from historical E3 RNG. Do not compare as exact reproduction.\'})\n    print(\'Сохранено:\', path, flush=True)\n\n@torch.inference_mode()\ndef smoke():\n    from e8_cityscapes import install, predict, token_mask\n    model = load_model(\'b5\'); state = install(model)\n    data = SegSet(\'cs_fit\'); img, _, name = data[0]\n    all_masks = {st: token_mask(np.ones((8,16),bool),st) for st in state}\n    plain = predict(model,state,img,{})\n    masked = predict(model,state,img,all_masks)\n    assert np.array_equal(plain,masked), \'All-true differs from unmasked model\'\n    path = CACHE/\'cs_fit\'/f\'{name}.npz\'\n    assert path.is_file(), \'Сначала выполните E0\'\n    with np.load(path) as z:\n        agreement = float((masked==z[\'pred_b5\']).mean())\n    assert agreement > .999, f\'Cached agreement={agreement}\'\n    selected = np.zeros((8,16),bool); selected[3:5,6:10] = True\n    routed = predict(model,state,img,{2:token_mask(selected,2)})\n    assert routed.shape == SIZE\n    write_json(RESULTS/\'smoke.json\', {\'complete\':True,\'all_true_equal\':True,\'cached_agreement\':agreement,\'selected_mask_forward_ok\':True})\n    print(\'Smoke OK. Совпадение с кэшем:\',agreement, flush=True)\n\n\ndef selftest():\n    from transformers import SegformerConfig, SegformerForSemanticSegmentation\n    from e8_cityscapes import install\n    torch.manual_seed(0)\n    cfg=SegformerConfig(num_channels=3,num_encoder_blocks=4,depths=[1,1,2,1],hidden_sizes=[8,16,32,64],num_attention_heads=[1,2,4,8],sr_ratios=[8,4,2,1],decoder_hidden_size=16,num_labels=19,drop_path_rate=0.)\n    m=SegformerForSemanticSegmentation(cfg).eval()\n    x=torch.randn(1,3,64,128)\n    with torch.inference_mode():\n        original=m(pixel_values=x).logits\n        states=install(m)\n        for st,s in states.items():\n            s.mask=torch.ones(1,(64//(4*2**st))*(128//(4*2**st)),1,dtype=torch.bool)\n        assert torch.equal(original,m(pixel_values=x).logits)\n        outputs=[]; handles=[]\n        for st,s in states.items():\n            s.mask=torch.zeros_like(s.mask)\n            for block in m.segformer.encoder.block[st]:\n                handles.append(block.register_forward_hook(lambda mod,inp,out:outputs.append(torch.equal(out[0],mod.state.entry))))\n        m(pixel_values=x)\n        for h in handles: h.remove()\n        assert outputs and all(outputs)\n    print(\'Tiny SegFormer: all-true / all-false OK\',flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'cache\',\'scores\',\'smoke\',\'e8\',\'selftest\']);args=p.parse_args()\n    if args.stage!=\'selftest\': assert torch.cuda.is_available(), \'Нужен GPU runtime\'\n    RESULTS.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n    import transformers\n    write_json(RESULTS/\'environment.json\', {\'signature\':SIGNATURE,\'python\':platform.python_version(),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\'size\':SIZE,\'tile\':TILE,\'domains\':DOMAINS,\'training\':False})\n    if args.stage==\'e8\':\n        from e8_cityscapes import main\n        main()\n    else: globals()[args.stage]()\n'}
for name, text in SOURCES.items():
    path = EXP/name
    if path.exists():
        assert path.read_text() == text, f'Изменён код: {path}. Для новой версии поменяй RUN_NAME.'
    else: path.write_text(text)
code_copy = SAVE/'source'
code_copy.mkdir(exist_ok=True)
for name in SOURCES: shutil.copy2(EXP/name, code_copy/name)
manifest = {'sha256':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
            'domains':['cs_fit','cs_test'], 'size':[512,1024], 'tile':64,
            'budgets':[0.1,0.2], 'frozen_models':True, 'rank_transductive':True}
manifest_path = SAVE/'run_config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Конфигурация изменилась: выбери новый RUN_NAME'
else: manifest_path.write_text(json.dumps(manifest, indent=2))
print('Код записан:', EXP)


In [ ]:
# 5. Восстановление после сброса VM и запускатель с логами.
import tarfile, time
for domain in ['cs_fit','cs_test']:
    backup = SAVE/'backups'/f'cache_{domain}.tar'
    if backup.is_file() and not (CACHE/domain).exists():
        print('Восстанавливаем:', backup, flush=True)
        with tarfile.open(backup) as tar:
            members = tar.getmembers()
            for m in members:
                assert m.isfile() or m.isdir()
                assert (CACHE/m.name).resolve().is_relative_to(CACHE.resolve())
            tar.extractall(CACHE)
for name in ['e8_knn_bank.npy','e8_tiles_cityscapes.npz']:
    src = SAVE/name
    if src.is_file() and not (CACHE/name).exists(): shutil.copy2(src,CACHE/name)

def run(stage):
    log_path = SAVE/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:', stage, '| Лог:', log_path, flush=True)
    started = time.monotonic()
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','runner.py',stage],cwd=EXP,
                                   env=os.environ.copy(), stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    tail = log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-started)/60:.1f} мин | '+ ' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait()
            raise
    print(log_path.read_text(errors='replace')[-8000:])
    assert status == 0, f'Ошибка {status}: смотри лог {log_path}'
    print(f'Завершено за {(time.monotonic()-started)/60:.1f} мин')


In [ ]:
# 6. Быстрая проверка API обёртки на маленькой случайной SegFormer, без загрузки весов.
run('selftest')


In [ ]:
# 7. Кэш B0/B5: batch=1, fp16, 500 кадров. Уже готовые записи проверяются и пропускаются.
# В конце каждого домена резервная копия кэша сохраняется на Drive.
run('cache')


In [ ]:
# 8. Проверка настоящей B5: all-true + один выбранный регион.
run('smoke')


In [ ]:
# 9. Банк kNN только из Frankfurt и признаки entropy+kNN для обоих доменов.
if (CACHE/'e8_tiles_cityscapes.npz').exists():
    print('Тайловые признаки уже сохранены — пропускаем.')
else:
    run('scores')
for name in ['e8_knn_bank.npy','e8_tiles_cityscapes.npz']:
    shutil.copy2(CACHE/name,SAVE/name)


## E8 — основной прогон
Восемь проходов B5 на каждый кадр: два метода × два бюджета × два варианта заморозки. Это самый долгий этап; ETA появится после первых пяти кадров. Выполненный домен сохраняется отдельно; при перезапуске незавершённый домен считается заново.

Метрика `full`, `s3`, `s3+s4` — **процент закрытого разрыва B5−B0**, а не mIoU. `agree` — совпадение с обычной B5 внутри выбранных тайлов. `rank` использует ранги всего домена; `oracle` использует GT и служит только ориентиром.

Текущий код считает полный кадр и затем маскирует обновления: **он не измеряет реальное ускорение**. Проверка пока на Cityscapes; перенос на ACDC — отдельный следующий этап без подгонки параметров на ACDC.


In [ ]:
# 10. Полный E8 на Cityscapes.
run('e8')


In [ ]:
# 11. Итоговая таблица и небольшой архив результатов (без тяжёлого кэша).
import csv
report_path = RESULTS/'e8_cityscapes.json'
report = json.loads(report_path.read_text())
rows = []
for domain, methods in report.items():
    for method, values in methods.items():
        rows.append({'domain':domain,'method_budget':method,**values})
assert rows, 'Результатов пока нет'
csv_path=RESULTS/'e8_cityscapes.csv'
with csv_path.open('w',newline='') as f:
    writer=csv.DictWriter(f, fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(rows))
archive=SAVE/'e8_cityscapes_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [RESULTS,SAVE/'source']:
        for p in folder.rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(SAVE))
    for p in list(SAVE.glob('*.log'))+[SAVE/'run_config.json',SAVE/'e8_knn_bank.npy',SAVE/'e8_tiles_cityscapes.npz']:
        if p.is_file(): z.write(p,p.name)
print('Сохранено на Drive:',archive)
print('Пришли e8_cityscapes.csv или e8_cityscapes_results.zip.')


## Если что-то остановилось
- **Нет файла ZIP**: исправь `ARCHIVE_DIR` в ячейке 2. Ноутбук не читает папку по публичной ссылке автоматически: нужен доступ через подключённый Drive.
- **CUDA out of memory**: проверь, что в GPU не осталось моделей из других ноутбуков; перезапусти среду и выполни ячейки снова. Установлен batch=1. Не снижай разрешение в середине запуска.
- **Ошибка transformers/torchvision**: пришли traceback; PyTorch/CUDA здесь не переустанавливаются.
- **Colab отключился**: подключись снова, выполни ячейки с начала. Архивы на Drive сохраняются, завершённые кэши доменов восстановятся, законченные домены E8 пропустятся. Незавершённый домен может потребовать повторного вычисления.
- **Нужна ACDC**: этот ноутбук намеренно Cityscapes-only. Сохрани банк `e8_knn_bank.npy`, конфигурации и результаты; позже применим те же опоры к ACDC.

Пятьсот изображений — официальная Cityscapes validation, внутреннее разделение по городам. Frankfurt используется для банка, не для независимой оценки переноса. Результаты этого ноутбука не подтверждают качество на ACDC.
